
extract_landmarks.ipynb
--------------------
Walk every A-Z class folder in data/asl_alphabet_train/, run MediaPipe Hands
on each image to extract 21 hand-landmark keypoints (x, y, z), normalise them
relative to the wrist, and save the result as a compressed NumPy archive at
data/landmarks.npz.

Skipped images (no hand detected) are counted and reported but not included in
the archive so training stays clean.

Usage:
    python src/extract_landmarks.py


In [5]:
#Imports
import os
import sys
import time
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

In [6]:
# Configuration

# Configuration
BASE_DIR       = os.path.abspath(os.path.join(os.getcwd(), ".."))

TRAIN_DIR      = os.path.join(BASE_DIR, "data", "asl_alphabet_train")
OUT_PATH       = os.path.join(BASE_DIR, "data", "landmarks.npz")
LANDMARK_MODEL = os.path.join(BASE_DIR, "models", "hand_landmarker.task")

# Only the 26 alphabetic classes (skip del / nothing / space)
CLASSES = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

NUM_LANDMARKS = 21
COORDS_PER_LM = 3   # x, y, z
FEATURE_DIM   = NUM_LANDMARKS * COORDS_PER_LM  # 63

print(f"BASE_DIR:       {BASE_DIR}")
print(f"TRAIN_DIR:      {TRAIN_DIR}")
print(f"LANDMARK_MODEL: {LANDMARK_MODEL}")

BASE_DIR:       /workspaces/asl-recognition
TRAIN_DIR:      /workspaces/asl-recognition/data/asl_alphabet_train
LANDMARK_MODEL: /workspaces/asl-recognition/models/hand_landmarker.task


In [ ]:
# Normalisation & extraction helpers

def normalise_landmarks(landmarks_list):
    """
    Given a flat list of 63 floats [x0,y0,z0, x1,y1,z1, ...], return a
    translation- and scale-invariant version:
      1. Subtract wrist (landmark 0) so the wrist is at the origin.
      2. Divide by the largest Euclidean distance from the wrist so the
         hand fits inside a unit sphere.
    """
    pts = np.array(landmarks_list, dtype=np.float32).reshape(NUM_LANDMARKS, COORDS_PER_LM)
    pts -= pts[0]
    scale = np.max(np.linalg.norm(pts, axis=1))
    if scale > 1e-6:
        pts /= scale
    return pts.flatten()


def _detect_hand(img_bgr, detector):
    """Run MediaPipe hand detection on a single BGR image. Returns raw coords or None."""
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result   = detector.detect(mp_image)
    if not result.hand_landmarks:
        return None
    raw = []
    for lm in result.hand_landmarks[0]:
        raw.extend([lm.x, lm.y, lm.z])
    return raw


def extract_with_fallbacks(img_bgr, detector):
    """Try multiple geometry-preserving preprocessing strategies to detect a hand.

    Returns normalised feature vector, or None if all strategies fail.
    Does NOT rotate or flip — rotation changes relative landmark positions
    (our normalization is translation+scale only, not rotation-invariant),
    which causes the model to confuse similar hand shapes (e.g. A→N, N→W).
    """
    # 1. Original image
    raw = _detect_hand(img_bgr, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    h, w = img_bgr.shape[:2]

    # 2. Bilateral filter (edge-preserving denoise — helps with fist poses like A, E, S)
    bilat = cv2.bilateralFilter(img_bgr, 9, 75, 75)
    raw = _detect_hand(bilat, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    # 3. Black-border padding (helps with tight crops)
    pad = int(max(h, w) * 0.15)
    padded = cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad,
                                cv2.BORDER_CONSTANT, value=(0, 0, 0))
    raw = _detect_hand(padded, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    # 4. White-border padding
    padded_w = cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad,
                                  cv2.BORDER_CONSTANT, value=(255, 255, 255))
    raw = _detect_hand(padded_w, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    # 5. CLAHE contrast enhancement
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    raw = _detect_hand(enhanced, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    # 6. Enhanced + padded (larger border for more detector context)
    pad2 = int(max(h, w) * 0.20)
    enhanced_padded = cv2.copyMakeBorder(enhanced, pad2, pad2, pad2, pad2,
                                         cv2.BORDER_CONSTANT, value=(0, 0, 0))
    raw = _detect_hand(enhanced_padded, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    # 7. Sharpened image (accentuates finger edges for fist-like poses)
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(img_bgr, -1, kernel)
    raw = _detect_hand(sharpened, detector)
    if raw is not None:
        return normalise_landmarks(raw)

    return None

In [8]:
def main():
    if not os.path.exists(LANDMARK_MODEL):
        sys.exit(f"[ERROR] Model file not found: {LANDMARK_MODEL}\n"
                 "Download it from:\n"
                 "  curl -Lo models/hand_landmarker.task "
                 "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task")

    features: list[np.ndarray] = []
    labels:   list[int]        = []
    skipped   = 0
    total_in  = 0

    base_opts = mp_python.BaseOptions(model_asset_path=LANDMARK_MODEL)
    options   = HandLandmarkerOptions(
        base_options=base_opts,
        running_mode=RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.05,
        min_hand_presence_confidence=0.05,
        min_tracking_confidence=0.05,
    )
    detector = HandLandmarker.create_from_options(options)

    t_start = time.time()

    for label_idx, cls in enumerate(CLASSES):
        cls_dir = os.path.join(TRAIN_DIR, cls)
        if not os.path.isdir(cls_dir):
            print(f"[WARN] Missing class directory: {cls_dir}", file=sys.stderr)
            continue

        img_files = [f for f in os.listdir(cls_dir)
                     if f.lower().endswith((".jpg", ".jpeg", ".png"))]

        cls_found = 0
        cls_skip  = 0

        for fname in img_files:
            total_in += 1
            img_path = os.path.join(cls_dir, fname)
            img_bgr  = cv2.imread(img_path)
            if img_bgr is None:
                cls_skip += 1
                skipped += 1
                continue

            feat = extract_with_fallbacks(img_bgr, detector)
            if feat is None:
                cls_skip += 1
                skipped  += 1
                continue

            features.append(feat)
            labels.append(label_idx)
            cls_found += 1

        elapsed = time.time() - t_start
        print(f"  [{cls}] detected={cls_found:4d}  skipped={cls_skip:4d}  "
              f"  elapsed={elapsed:6.1f}s")

    detector.close()

    X = np.array(features, dtype=np.float32)
    y = np.array(labels,   dtype=np.int32)

    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    np.savez_compressed(OUT_PATH, X=X, y=y, classes=np.array(CLASSES))

    print(f"\nDone.  Saved {X.shape[0]:,} samples  ({skipped:,} skipped / {total_in:,} total)")
    print(f"Feature shape: {X.shape}   Label shape: {y.shape}")
    print(f"Output: {os.path.abspath(OUT_PATH)}")


main()

W0000 00:00:1775225276.485791   38254 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775225276.584061   38261 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  [A] detected=2912  skipped=  88    elapsed= 151.6s
  [B] detected=2720  skipped= 280    elapsed= 329.2s
  [C] detected=2896  skipped= 104    elapsed= 488.4s
  [D] detected=2981  skipped=  19    elapsed= 628.9s
  [E] detected=2850  skipped= 150    elapsed= 801.1s
  [F] detected=3000  skipped=   0    elapsed= 931.3s
  [G] detected=2945  skipped=  55    elapsed=1077.7s
  [H] detected=2902  skipped=  98    elapsed=1234.2s
  [I] detected=2805  skipped= 195    elapsed=1402.6s
  [J] detected=2977  skipped=  23    elapsed=1541.9s
  [K] detected=2848  skipped= 152    elapsed=1690.7s
  [L] detected=2925  skipped=  75    elapsed=1834.1s
  [M] detected=2951  skipped=  49    elapsed=1989.3s
  [N] detected=2876  skipped= 124    elapsed=2166.3s
  [O] detected=2962  skipped=  38    elapsed=2312.7s
  [P] detected=2849  skipped= 151    elapsed=2478.5s
  [Q] detected=2877  skipped= 123    elapsed=2632.1s
  [R] detected=2941  skipped=  59    elapsed=2775.8s
  [S] detected=2925  skipped=  75    elapsed=2